# LeetCode 535 — Encode and Decode TinyURL bằng Python

Link đề: https://leetcode.com/problems/encode-and-decode-tinyurl/ · Độ khó: **Medium** · Chủ đề: Design, Hash Table, String, Hash Function

Skeleton LeetCode (Python):

```python
class Codec:

    def encode(self, longUrl: str) -> str:
        """Encodes a URL to a shortened URL.
        """

    def decode(self, shortUrl: str) -> str:
        """Decodes a shortened URL to its original URL.
        """


# Your Codec object will be instantiated and called as such:
# codec = Codec()
# codec.decode(codec.encode(url))
```

Ghi chú: bài **thiết kế lớp** và **đáp án không duy nhất** — mỗi cách cài đặt cho ra một short URL khác nhau (miễn là `decode(encode(url)) == url`). Vì vậy notebook **không so đáp án cứng** mà test bằng **round-trip** (`decode(encode(url)) == url`) và các **tính chất phụ** (short URL khác nhau cho URL khác nhau, nhiều URL trên cùng một đối tượng không đè nhau, ...). Tên lớp trong skeleton Python của LeetCode là `Codec` (một số bản mô tả ghi `Solution`) — nên **đối chiếu lại đề gốc** khi nộp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thiết kế lớp `Codec` có `encode(longUrl)` → short URL và `decode(shortUrl)` → URL gốc, **không bị ràng buộc cách mã hóa**, chỉ cần `decode(encode(url)) == url` trên **cùng một đối tượng**.
2. Nắm công cụ Python cần thiết: lớp với **trạng thái** (`self.x` trong `__init__`), `dict`, `str.rsplit("/", 1)`, f-string, đổi cơ số bằng `divmod`, `random.choices`, `hashlib.sha256`, `int.from_bytes`, `base64`/`zlib`.
3. Đi từ **danh sách + tìm tuyến tính** → **bộ đếm id + dict** → **mã ngẫu nhiên 6 ký tự có thử lại khi đụng độ** → **băm SHA-256 + dò thêm "salt" khi đụng độ** → **mã hóa không trạng thái (nén + base64)**.
4. Hiểu mấu chốt: bài thực chất là "**gán cho mỗi URL một khóa duy nhất**". Duy nhất **bằng cấu tạo** (bộ đếm) thì an toàn tuyệt đối; duy nhất **bằng may rủi** (ngẫu nhiên/băm) thì **bắt buộc** có xử lý đụng độ (collision).
5. Biết cách **tự dựng bộ kiểm tra** cho bài thiết kế có nhiều đáp án đúng: round-trip, tính duy nhất, tính độc lập giữa các đối tượng, thống kê có seed, ép đụng độ để thử nhánh hiếm; và tránh các lỗi kinh điển (cắt mã cố định độ dài, base64 chuẩn chứa `/`, băm mà không xử lý đụng độ, ...).

## 2. Đề bài & ví dụ

TinyURL là dịch vụ rút gọn đường dẫn: bạn nhập `https://leetcode.com/problems/design-tinyurl` và nhận về một đường dẫn ngắn như `http://tinyurl.com/4e9iAk`. Hãy thiết kế lớp `Codec` gồm:

- `Codec()`: khởi tạo hệ thống.
- `encode(longUrl)`: trả về một **short URL** cho `longUrl`.
- `decode(shortUrl)`: trả về **URL gốc**. Đảm bảo `shortUrl` do **chính đối tượng này** `encode` ra.

**Không có ràng buộc** nào về cách thuật toán hoạt động — chỉ cần URL mã hóa được rồi giải mã về đúng URL ban đầu.

**Ví dụ:**

```
Input:  url = "https://leetcode.com/problems/design-tinyurl"
Output: "https://leetcode.com/problems/design-tinyurl"

codec = Codec()
tiny = codec.encode(url)      # trả về short URL (tùy cài đặt)
ans  = codec.decode(tiny)     # phải trả về đúng url ban đầu
```

(LeetCode chạy đúng chuỗi lệnh trên rồi so `ans` với `url`; short URL cụ thể **không** được so.)

**Ràng buộc:** `1 <= url.length <= 10^4`; `url` đảm bảo là URL hợp lệ.

In [1]:
import base64
import hashlib
import random
import string
import subprocess
import sys
import time
import zlib
from collections import Counter


URL0 = "https://leetcode.com/problems/design-tinyurl"


def leetcode_driver(cls, url):
    """Mô phỏng cách LeetCode chạy: codec = Codec(); ans = codec.decode(codec.encode(url))."""
    codec = cls()
    tiny = codec.encode(url)
    ans = codec.decode(tiny)
    return tiny, ans


print("URL gốc:", URL0)
print("độ dài :", len(URL0))

URL gốc: https://leetcode.com/problems/design-tinyurl
độ dài : 44


## 3. Kiến thức nền

### 3.1. Lớp có **trạng thái**: `__init__` và `self`

`Codec()` chạy `__init__` **một lần**; các thuộc tính gắn vào `self` (ví dụ `self.m = {}`) **sống tiếp** giữa các lần gọi `encode`/`decode` trên cùng đối tượng. Chính nhờ đó `decode` "nhớ" được những gì `encode` đã làm. Hai đối tượng khác nhau có trạng thái **riêng**.

In [2]:
class Counter3:
    def __init__(self):
        self.n = 0                    # trạng thái riêng của từng đối tượng

    def hit(self):
        self.n += 1
        return self.n


a, b = Counter3(), Counter3()
print(a.hit(), a.hit(), a.hit(), b.hit())    # 1 2 3 1: a và b độc lập

1 2 3 1


### 3.2. `dict`: tra cứu theo khóa trong `O(1)` trung bình

`d[key] = value` ghi, `d[key]` đọc (khóa không có thì `KeyError`), `key in d` kiểm tra, `d.get(key)` trả `None` nếu thiếu. Hai dict "ngược nhau" (`url → code` và `code → url`) cho phép tra cứu **cả hai chiều** nhanh.

In [3]:
d = {"1": "https://a.com", "2": "https://b.com"}
print(d["1"], "2" in d, "9" in d, d.get("9"))
try:
    d["9"]
except KeyError as e:
    print("KeyError:", e)

rev = {v: k for k, v in d.items()}           # đảo chiều dict
print(rev)

https://a.com True False None
KeyError: '9'
{'https://a.com': '1', 'https://b.com': '2'}


### 3.3. Xử lý chuỗi: f-string, `split`, `rsplit`

- f-string: `f"{domain}{idx}"` ghép chuỗi.
- `s.split("/")` cắt theo `/` thành list; `[-1]` lấy phần cuối.
- `s.rsplit("/", 1)` cắt **một lần từ bên phải** — an toàn hơn khi phần cuối chính là "mã" cần lấy, vì không quan tâm có bao nhiêu dấu `/` ở phần trước.

In [4]:
domain = "http://tinyurl.com/"
short = f"{domain}{42}"
print(short)
print(short.split("/"))
print(short.split("/")[-1], short.rsplit("/", 1)[-1])
print(URL0.rsplit("/", 1))                   # ['https://leetcode.com/problems', 'design-tinyurl']

http://tinyurl.com/42
['http:', '', 'tinyurl.com', '42']
42 42
['https://leetcode.com/problems', 'design-tinyurl']


### 3.4. Đổi cơ số bằng `divmod` (mã ngắn kiểu base-62)

Số `n` viết theo cơ số `b`: lặp `n, r = divmod(n, b)` lấy phần dư `r` làm chữ số, từ thấp đến cao, rồi đảo ngược. Dùng bảng chữ cái 62 ký tự `0-9a-zA-Z` thì số lớn được viết rất ngắn: `62^6 ≈ 56,8 tỷ` — chỉ cần 6 ký tự.

In [5]:
ALPHABET = string.digits + string.ascii_lowercase + string.ascii_uppercase     # 62 ký tự
print(len(ALPHABET), ALPHABET)


def to_base62(n):
    if n == 0:
        return ALPHABET[0]
    out = []
    while n:
        n, r = divmod(n, 62)
        out.append(ALPHABET[r])
    return "".join(reversed(out))


def from_base62(s):
    n = 0
    for ch in s:
        n = n * 62 + ALPHABET.index(ch)
    return n


for n in (0, 1, 61, 62, 125, 3844, 10**6):
    print(n, "->", to_base62(n), "->", from_base62(to_base62(n)))
    assert from_base62(to_base62(n)) == n
print("62**6 =", 62**6)

62 0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
0 -> 0 -> 0
1 -> 1 -> 1
61 -> Z -> 61
62 -> 10 -> 62
125 -> 21 -> 125
3844 -> 100 -> 3844
1000000 -> 4c92 -> 1000000
62**6 = 56800235584


### 3.5. `random`: chọn ký tự ngẫu nhiên, có seed

`random.choices(seq, k=6)` chọn **6 phần tử có lặp**. Dùng `random.Random(seed)` để có bộ sinh riêng, **tái lập được** khi test; không truyền seed thì mỗi lần chạy một khác.

In [6]:
rng = random.Random(7)
print("".join(rng.choices(ALPHABET, k=6)), "".join(rng.choices(ALPHABET, k=6)))
rng = random.Random(7)
print("".join(rng.choices(ALPHABET, k=6)), "(cùng seed -> cùng kết quả)")

k9E4xm 3v2q45
k9E4xm (cùng seed -> cùng kết quả)


### 3.5b. Xác suất đụng độ: nghịch lý ngày sinh (birthday paradox)

Nếu mỗi URL lấy một mã ngẫu nhiên trong `N` khả năng thì sau `n` URL, xác suất có **ít nhất một cặp trùng** xấp xỉ `n² / (2N)` (khi nhỏ). Với `N = 62^6 ≈ 5,7·10^10` và `n = 10^5` là khoảng `10^10 / 1,1·10^11 ≈ 9%` — **không nhỏ**. Vì thế ngẫu nhiên/băm **luôn** cần xử lý đụng độ.

In [7]:
N = 62 ** 6
for n in (10**3, 10**4, 10**5, 10**6):
    print(f"n={n:>8}: xác suất đụng độ ≈ {min(1, n * n / (2 * N)):.5f}")

n=    1000: xác suất đụng độ ≈ 0.00001
n=   10000: xác suất đụng độ ≈ 0.00088
n=  100000: xác suất đụng độ ≈ 0.08803
n= 1000000: xác suất đụng độ ≈ 1.00000


### 3.6. Băm: `hashlib.sha256` và `int.from_bytes`

Hàm băm biến chuỗi bất kỳ thành dãy byte có độ dài cố định; **cùng đầu vào → cùng đầu ra**, nhưng **không đảo ngược được**, và hai chuỗi khác nhau **có thể** trùng (hiếm). `int.from_bytes(b, "big")` đổi dãy byte thành số nguyên để lấy phần dư/đổi cơ số.

(Hàm `hash()` có sẵn của Python cho `str` bị **xáo hạt giống mỗi lần chạy chương trình**, nên **không** dùng để sinh mã ổn định.)

In [8]:
d1 = hashlib.sha256(URL0.encode()).digest()
d2 = hashlib.sha256(URL0.encode()).digest()
print(len(d1), d1 == d2, d1.hex()[:16])
num = int.from_bytes(d1, "big")
print(num % 62**6, to_base62(num % 62**6).rjust(6, "0"))

32 True f86ef16a6a7d7d6b
30231764676 wZXp1G


### 3.7. `base64` và `zlib`: mã hóa **đảo ngược được** (không cần nhớ gì)

`zlib.compress` nén byte; `base64.urlsafe_b64encode` đổi byte thành chuỗi gồm `A-Za-z0-9-_=` (bản "url-safe" **không** có `/` và `+`). Giải mã làm ngược lại. Đầu ra chứa đủ thông tin để khôi phục dữ liệu — không cần bảng tra.

In [9]:
raw = URL0.encode()
z = zlib.compress(raw)
enc = base64.urlsafe_b64encode(z).decode()
print(len(raw), len(z), len(enc))
print(enc)
pad = "=" * (-len(enc) % 4)
assert zlib.decompress(base64.urlsafe_b64decode(enc + pad)) == raw
print("khôi phục đúng:", zlib.decompress(base64.urlsafe_b64decode(enc + pad)).decode())

44 52 72
eJzLKCkpKLbS189JTS1Jzk9J1UvOz9UvKMpPyknNLdZPSS3OTM_TLcnMqywtygEAdlcQ_g==
khôi phục đúng: https://leetcode.com/problems/design-tinyurl


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng một **cuốn sổ ở quầy lễ tân**. Khách đưa một địa chỉ rất dài; bạn **ghi vào dòng số 1** và đưa lại tấm phiếu "`tinyurl.com/1`". Khách tiếp theo đưa địa chỉ khác: bạn ghi vào dòng 2 và đưa phiếu "`tinyurl.com/2`". Khi ai đó mang phiếu `.../2` tới, bạn đọc số `2` rồi mở **dòng 2** của sổ, đọc ra địa chỉ.

Tính tay:

| Gọi | Cuốn sổ sau khi gọi | Trả về |
|---|---|---|
| `encode("https://a.com/x")` | `{1: "https://a.com/x"}` | `tinyurl.com/1` |
| `encode("https://b.com/y")` | `{1: ..., 2: "https://b.com/y"}` | `tinyurl.com/2` |
| `decode("tinyurl.com/2")` | — | `"https://b.com/y"` |

Số thứ tự **không bao giờ trùng** vì luôn tăng — nên mỗi phiếu chỉ ứng với **đúng một** dòng. Mọi cách giải của bài đều là biến thể của "cuốn sổ" này; điểm khác nhau chỉ là **số trên phiếu được tạo ra thế nào** (đếm, bốc thăm, băm) hoặc thậm chí **nhét cả địa chỉ vào phiếu** để khỏi cần sổ.

## 5. Cách 1 — Brute Force / baseline: danh sách + tìm tuyến tính

**Ý tưởng:** giữ một `list` các URL đã gặp. `encode(url)`: nếu `url` đã có trong list thì lấy chỉ số cũ, nếu chưa thì `append` rồi lấy chỉ số mới; trả về `domain + str(chỉ_số)`. `decode`: đọc chỉ số ở cuối short URL rồi `urls[chỉ_số]`.

Đây là baseline "đơn giản nhất có thể đúng": chỉ số chính là số dòng của cuốn sổ, và nhờ tìm trùng nên **cùng một URL luôn ra cùng một short URL** (tính chất đẹp, đề không yêu cầu).

In [10]:
class CodecList:
    DOMAIN = "http://tinyurl.com/"

    def __init__(self):
        self.urls = []

    def encode(self, longUrl):
        if longUrl in self.urls:                 # tìm tuyến tính: O(n) so sánh chuỗi
            idx = self.urls.index(longUrl)
        else:
            idx = len(self.urls)
            self.urls.append(longUrl)
        return self.DOMAIN + str(idx)

    def decode(self, shortUrl):
        idx = int(shortUrl.rsplit("/", 1)[-1])
        return self.urls[idx]


tiny, ans = leetcode_driver(CodecList, URL0)
print(tiny, "->", ans)
assert ans == URL0

http://tinyurl.com/0 -> https://leetcode.com/problems/design-tinyurl


**Độ phức tạp:** `encode` tốn `O(n·L)` trong trường hợp xấu (`n` URL đã lưu, mỗi lần so chuỗi tới `L` ký tự — thực tế so lệch sớm nên nhanh hơn), `decode` `O(1)` (+ `O(L)` tùy cách tính). Space `O(tổng độ dài URL)`.

**Nút thắt:** để đảm bảo "cùng URL ↔ cùng short URL" ta phải **quét cả danh sách mỗi lần `encode`** — chậm dần khi lưu nhiều URL. Mà đề **không** đòi cùng URL cho cùng short URL! Bỏ yêu cầu thừa đó đi, ta chỉ còn cần "mỗi lần `encode` cấp một khóa mới" — chính là bộ đếm.

## 6. Ý tưởng thuật toán / bước đột phá

`decode` chỉ cần một ánh xạ **short → long**. Có hai họ cách tạo ánh xạ đó:

1. **Có trạng thái (nhớ):** lưu `code → url` trong `dict`. Vấn đề duy nhất là **chọn `code` sao cho không trùng** mã đã cấp:
   - **Bằng cấu tạo:** dùng **bộ đếm tăng dần** — không bao giờ trùng, không cần kiểm tra. (Cách 2)
   - **Bằng may rủi:** bốc mã **ngẫu nhiên** hoặc **băm** URL rồi **kiểm tra trùng** và thử lại/đổi "salt" nếu đụng. (Cách 3, Cách 4)
2. **Không trạng thái (tính):** để chính short URL **chứa** URL gốc dưới dạng đã nén/mã hóa; `decode` giải ngược lại, không cần dict. (Cách 5)

Điểm mấu chốt về độ tin cậy: **bất biến** "với mọi `code` đã cấp, `m[code]` đúng là URL được cấp mã đó" được giữ **tự động** nếu mã luôn mới (bộ đếm), nhưng phải **ép giữ** (kiểm tra `code not in m`) nếu mã đến từ ngẫu nhiên/băm.

## 7. Các cách tiếp theo

Tổng cộng 5 cách: Cách 1 (danh sách tuyến tính, mục 5) và bốn cách dưới đây. Về ý tưởng, chúng khác nhau thật: **đếm** (duy nhất bằng cấu tạo), **ngẫu nhiên** (duy nhất bằng kiểm tra), **băm** (mã xác định theo nội dung + dò khi đụng), **không trạng thái** (mã chứa dữ liệu).

### Cách 2 — Bộ đếm id + dict (cách chính thức, nhanh và an toàn)

**Giải thích từng đoạn:**

1. `self.idx` đếm số URL đã mã hóa; `self.m` là dict `id (chuỗi) → URL`.
2. `encode`: tăng `idx`, lưu `m[str(idx)] = longUrl`, trả `domain + idx`.
3. `decode`: lấy đoạn sau dấu `/` cuối, tra `m`.

Mọi lệnh `O(1)` (trừ việc băm/ sao chép chuỗi URL dài `L`).

In [11]:
class CodecCounter:
    DOMAIN = "http://tinyurl.com/"

    def __init__(self):
        self.m = {}
        self.idx = 0

    def encode(self, longUrl):
        self.idx += 1
        self.m[str(self.idx)] = longUrl
        return f"{self.DOMAIN}{self.idx}"

    def decode(self, shortUrl):
        return self.m[shortUrl.rsplit("/", 1)[-1]]


tiny, ans = leetcode_driver(CodecCounter, URL0)
print(tiny, "->", ans)
assert ans == URL0

http://tinyurl.com/1 -> https://leetcode.com/problems/design-tinyurl


**Code nộp LeetCode (Cách 2):**

```python
class Codec:
    def __init__(self):
        self.m = {}
        self.idx = 0
        self.domain = "https://tinyurl.com/"

    def encode(self, longUrl: str) -> str:
        """Encodes a URL to a shortened URL."""
        self.idx += 1
        self.m[str(self.idx)] = longUrl
        return f"{self.domain}{self.idx}"

    def decode(self, shortUrl: str) -> str:
        """Decodes a shortened URL to its original URL."""
        return self.m[shortUrl.rsplit("/", 1)[-1]]
```

**Độ phức tạp:** `encode`/`decode` đều `O(L)` (`L` = độ dài URL, do băm/ghi chuỗi; coi như `O(1)` với `L ≤ 10^4`). Space `O(tổng độ dài các URL đã mã hóa)`.

### Cách 3 — Mã ngẫu nhiên 6 ký tự + thử lại khi đụng độ

**Giải thích từng đoạn:**

1. Bảng chữ cái 62 ký tự; mã dài `LEN = 6` ⇒ `62^6 ≈ 56,8 tỷ` mã.
2. `encode`: bốc `LEN` ký tự ngẫu nhiên thành `code`; **nếu `code` đã có trong `m` thì bốc lại** (vòng `while`). Khi chưa có thì gán `m[code] = longUrl`.
3. `decode`: tra `m` bằng đoạn sau dấu `/` cuối.

`ALPHABET`, `LEN` đặt thành **thuộc tính lớp** để lát nữa test có thể ép bảng chữ cái rất nhỏ nhằm **gây đụng độ có chủ đích**. Bộ sinh `random.Random(seed)` riêng của từng đối tượng giúp test tái lập (không truyền seed thì mỗi lần một khác).

In [12]:
class CodecRandom:
    DOMAIN = "http://tinyurl.com/"
    ALPHA = ALPHABET
    LEN = 6

    def __init__(self, seed=None):
        self.m = {}
        self.rng = random.Random(seed)
        self.retries = 0                                  # đếm số lần phải bốc lại (để quan sát)

    def encode(self, longUrl):
        while True:
            code = "".join(self.rng.choices(self.ALPHA, k=self.LEN))
            if code not in self.m:
                break
            self.retries += 1                             # đụng độ: bốc lại
        self.m[code] = longUrl
        return self.DOMAIN + code

    def decode(self, shortUrl):
        return self.m[shortUrl.rsplit("/", 1)[-1]]


c = CodecRandom(seed=1)
tiny = c.encode(URL0)
print(tiny, "->", c.decode(tiny))
assert c.decode(tiny) == URL0

http://tinyurl.com/8QLfur -> https://leetcode.com/problems/design-tinyurl


**Code nộp LeetCode (Cách 3):**

```python
import random
import string

class Codec:
    ALPHA = string.digits + string.ascii_letters      # 62 ký tự

    def __init__(self):
        self.m = {}

    def encode(self, longUrl: str) -> str:
        while True:
            code = "".join(random.choices(self.ALPHA, k=6))
            if code not in self.m:
                break
        self.m[code] = longUrl
        return "http://tinyurl.com/" + code

    def decode(self, shortUrl: str) -> str:
        return self.m[shortUrl.rsplit("/", 1)[-1]]
```

**Độ phức tạp:** mỗi lần bốc thành công với xác suất `1 − n/62^6`; kỳ vọng số lần bốc `≈ 1 + n/62^6 ≈ 1` ⇒ `encode` `O(L + 6)` kỳ vọng. Space `O(tổng độ dài URL)`. **Rủi ro:** nếu số URL tiến gần `62^6` thì vòng `while` kéo dài; trong thực tế ta tăng `LEN`.

### Cách 4 — Băm SHA-256 + dò thêm "salt" khi đụng độ

**Giải thích từng đoạn:**

1. Mã **xác định theo nội dung**: `code(url, salt) = base62( SHA256(f"{salt}:{url}") mod 62^6 )`, đệm `0` để đủ 6 ký tự.
2. `encode` thử `salt = 0, 1, 2, ...`:
   - `code` **chưa có** trong `m` ⇒ cấp mã này, lưu `m[code] = url`.
   - `code` đã có **và** trỏ đúng `url` này ⇒ chính URL này đã được mã hóa trước đó ⇒ trả lại mã cũ (**cùng URL → cùng short URL**, miễn phí!).
   - `code` đã có nhưng trỏ **URL khác** ⇒ đụng độ ⇒ tăng `salt` thử tiếp.
3. `decode` y hệt các cách trước.

Vì thứ tự thử `salt` là **xác định**, URL cũ luôn tìm lại đúng mã đã cấp.

In [13]:
class CodecHash:
    DOMAIN = "http://tinyurl.com/"
    LEN = 6

    def __init__(self):
        self.m = {}
        self.probes = 0                                   # số lần phải đổi salt (để quan sát)

    def _code(self, url, salt):
        digest = hashlib.sha256(f"{salt}:{url}".encode()).digest()
        n = int.from_bytes(digest, "big") % (62 ** self.LEN)
        return to_base62(n).rjust(self.LEN, ALPHABET[0])

    def encode(self, longUrl):
        salt = 0
        while True:
            code = self._code(longUrl, salt)
            if code not in self.m:                        # ô trống: cấp mới
                self.m[code] = longUrl
                return self.DOMAIN + code
            if self.m[code] == longUrl:                   # chính URL này: dùng lại mã cũ
                return self.DOMAIN + code
            salt += 1                                     # đụng độ với URL khác: đổi salt
            self.probes += 1

    def decode(self, shortUrl):
        return self.m[shortUrl.rsplit("/", 1)[-1]]


tiny, ans = leetcode_driver(CodecHash, URL0)
print(tiny, "->", ans)
assert ans == URL0
c = CodecHash()
assert c.encode(URL0) == c.encode(URL0), "cùng URL phải ra cùng short URL"

http://tinyurl.com/FPltjS -> https://leetcode.com/problems/design-tinyurl


**Code nộp LeetCode (Cách 4):**

```python
import hashlib, string

class Codec:
    ALPHA = string.digits + string.ascii_letters

    def __init__(self):
        self.m = {}

    def _code(self, url: str, salt: int) -> str:
        n = int.from_bytes(hashlib.sha256(f"{salt}:{url}".encode()).digest(), "big") % (62 ** 6)
        out = []
        for _ in range(6):
            n, r = divmod(n, 62)
            out.append(self.ALPHA[r])
        return "".join(out)

    def encode(self, longUrl: str) -> str:
        salt = 0
        while True:
            code = self._code(longUrl, salt)
            if code not in self.m:
                self.m[code] = longUrl
                return "http://tinyurl.com/" + code
            if self.m[code] == longUrl:
                return "http://tinyurl.com/" + code
            salt += 1

    def decode(self, shortUrl: str) -> str:
        return self.m[shortUrl.rsplit("/", 1)[-1]]
```

**Độ phức tạp:** mỗi lần thử tốn `O(L)` để băm; số lần thử kỳ vọng `≈ 1`. Tổng `encode` `O(L)` kỳ vọng. Space `O(tổng độ dài URL)`.

### Cách 5 — Mã hóa **không trạng thái**: nén + base64 chứa luôn URL

**Giải thích từng đoạn:**

1. `encode`: `zlib.compress(url)` → `base64.urlsafe_b64encode` → bỏ ký tự đệm `=` → ghép sau domain. **Không lưu gì trong đối tượng.**
2. `decode`: lấy đoạn sau `/` cuối, bù lại phần đệm `=` cho đủ bội số của 4, giải base64 rồi `zlib.decompress`.

Hệ quả thú vị: `decode` hoạt động **ngay cả trên một đối tượng `Codec` mới tinh** (vì thông tin nằm ở short URL). Đổi lại, short URL **không ngắn** cho URL ngẫu nhiên (nén không giúp gì), nên chỉ phù hợp để minh họa "đề không ràng buộc cách mã hóa".

In [14]:
class CodecStateless:
    DOMAIN = "http://tinyurl.com/"

    def encode(self, longUrl):
        z = zlib.compress(longUrl.encode("utf-8"))
        return self.DOMAIN + base64.urlsafe_b64encode(z).decode().rstrip("=")

    def decode(self, shortUrl):
        code = shortUrl.rsplit("/", 1)[-1]
        code += "=" * (-len(code) % 4)                    # bù lại phần đệm đã bỏ
        return zlib.decompress(base64.urlsafe_b64decode(code)).decode("utf-8")


tiny, ans = leetcode_driver(CodecStateless, URL0)
print(tiny, "->", ans)
assert ans == URL0
fresh = CodecStateless()                                  # đối tượng KHÁC vẫn giải mã được
assert fresh.decode(tiny) == URL0
print("đối tượng mới decode được:", fresh.decode(tiny) == URL0)

http://tinyurl.com/eJzLKCkpKLbS189JTS1Jzk9J1UvOz9UvKMpPyknNLdZPSS3OTM_TLcnMqywtygEAdlcQ_g -> https://leetcode.com/problems/design-tinyurl
đối tượng mới decode được: True


**Code nộp LeetCode (Cách 5):**

```python
import base64, zlib

class Codec:
    def encode(self, longUrl: str) -> str:
        z = zlib.compress(longUrl.encode("utf-8"))
        return "http://tinyurl.com/" + base64.urlsafe_b64encode(z).decode().rstrip("=")

    def decode(self, shortUrl: str) -> str:
        code = shortUrl.rsplit("/", 1)[-1]
        code += "=" * (-len(code) % 4)
        return zlib.decompress(base64.urlsafe_b64decode(code)).decode("utf-8")
```

**Độ phức tạp:** `encode`/`decode` `O(L)`; Space phụ `O(1)` trạng thái (đầu ra dài `O(L)`). Short URL **dài** cỡ `~1,33·L'` với `L'` là độ dài sau nén.

### So sánh nhanh bốn "cách tạo mã" trên cùng một URL

In [15]:
for cls in (CodecList, CodecCounter, CodecRandom, CodecHash, CodecStateless):
    c = cls()
    tiny = c.encode(URL0)
    print(f"{cls.__name__:<15} {tiny}   (dài {len(tiny)})")

CodecList       http://tinyurl.com/0   (dài 20)
CodecCounter    http://tinyurl.com/1   (dài 20)
CodecRandom     http://tinyurl.com/6ke9JS   (dài 25)
CodecHash       http://tinyurl.com/FPltjS   (dài 25)
CodecStateless  http://tinyurl.com/eJzLKCkpKLbS189JTS1Jzk9J1UvOz9UvKMpPyknNLdZPSS3OTM_TLcnMqywtygEAdlcQ_g   (dài 89)


## 8. TRACE từng bước

**Trace Cách 2 (bộ đếm):** ba lệnh `encode`, một lệnh `encode` trùng URL, rồi `decode`.

| Bước | Lệnh | `idx` sau lệnh | `m` sau lệnh | Trả về |
|---|---|---|---|---|
| 0 | `Codec()` | 0 | `{}` | — |
| 1 | `encode("https://a.com")` | 1 | `{"1": "https://a.com"}` | `http://tinyurl.com/1` |
| 2 | `encode("https://b.com")` | 2 | `{"1": a, "2": b}` | `http://tinyurl.com/2` |
| 3 | `encode("https://a.com")` (trùng) | 3 | `{"1": a, "2": b, "3": a}` | `http://tinyurl.com/3` |
| 4 | `decode(".../2")` | 3 | không đổi | `"https://b.com"` |
| 5 | `decode(".../3")` | 3 | không đổi | `"https://a.com"` |

Cách 2 coi lần thứ 3 là URL mới (không dedupe) — vẫn đúng đề vì cả `/1` lẫn `/3` đều `decode` về `a`. Hàm `trace_counter` in từng lệnh:

In [16]:
def trace_counter(script):
    codec = CodecCounter()
    print(f"{'lệnh':<28}{'idx':<5}{'trả về'}")
    for op, arg in script:
        if op == "encode":
            out = codec.encode(arg)
            print(f"{'encode(' + repr(arg) + ')':<28}{codec.idx:<5}{out}")
        else:
            out = codec.decode(arg)
            print(f"{'decode(...' + arg[-2:] + ')':<28}{codec.idx:<5}{out!r}")
    print("m =", codec.m, "\n")
    return codec


c = trace_counter([("encode", "https://a.com"), ("encode", "https://b.com"),
                   ("encode", "https://a.com"), ("decode", "http://tinyurl.com/2"),
                   ("decode", "http://tinyurl.com/3")])
assert c.m == {"1": "https://a.com", "2": "https://b.com", "3": "https://a.com"}

lệnh                        idx  trả về
encode('https://a.com')     1    http://tinyurl.com/1
encode('https://b.com')     2    http://tinyurl.com/2
encode('https://a.com')     3    http://tinyurl.com/3
decode(.../2)               3    'https://b.com'
decode(.../3)               3    'https://a.com'
m = {'1': 'https://a.com', '2': 'https://b.com', '3': 'https://a.com'} 



**Trace Cách 3 (ngẫu nhiên) với bảng chữ cái chỉ có `"ab"` và mã dài `1`** — chỉ có **2 mã** nên đụng độ xảy ra thường xuyên; xem vòng `while` bốc lại thế nào (seed cố định để tái lập):

In [17]:
class CodecRandomTiny(CodecRandom):
    ALPHA = "ab"
    LEN = 1


def trace_random(urls, seed):
    c = CodecRandomTiny(seed)
    rng = random.Random(seed)                              # bản sao bộ sinh để in từng lần bốc
    m = {}
    for u in urls:
        attempts = []
        while True:
            code = "".join(rng.choices("ab", k=1))
            attempts.append(code)
            if code not in m:
                break
        m[code] = u
        print(f"encode({u}): các lần bốc {attempts} -> dùng mã {code!r}; m = {m}")
        assert c.encode(u).endswith("/" + code)
    return m


m = trace_random(["u1", "u2"], seed=3)
assert sorted(m) == ["a", "b"]

encode(u1): các lần bốc ['a'] -> dùng mã 'a'; m = {'a': 'u1'}
encode(u2): các lần bốc ['b'] -> dùng mã 'b'; m = {'a': 'u1', 'b': 'u2'}


**Trace Cách 4 (băm + salt)** ép đụng độ bằng cách cho mã chỉ dài 1 ký tự (62 ô). Mỗi lần thử in `salt`, mã và kết luận:

In [18]:
class CodecHashTiny(CodecHash):
    LEN = 1                                                # chỉ 62 mã: đụng độ rất sớm


def trace_hash(urls):
    c = CodecHashTiny()
    for u in urls:
        salt = 0
        while True:
            code = c._code(u, salt)
            if code not in c.m:
                verdict = "ô trống -> cấp mới"
                done = True
            elif c.m[code] == u:
                verdict = "đã có chính URL này -> dùng lại"
                done = True
            else:
                verdict = f"đụng độ với {c.m[code]!r} -> salt+1"
                done = False
            print(f"  url={u!r:<6} salt={salt} code={code!r}  {verdict}")
            if done:
                break
            salt += 1
        short = c.encode(u)
        print(f"=> encode({u!r}) = {short}")
    return c


urls_t = [f"u{i}" for i in range(12)]
c = trace_hash(urls_t)
print("probes tổng cộng:", c.probes)
for u in urls_t:
    assert c.decode(c.encode(u)) == u
assert c.probes > 0

  url='u0'   salt=0 code='m'  ô trống -> cấp mới
=> encode('u0') = http://tinyurl.com/m
  url='u1'   salt=0 code='a'  ô trống -> cấp mới
=> encode('u1') = http://tinyurl.com/a
  url='u2'   salt=0 code='z'  ô trống -> cấp mới
=> encode('u2') = http://tinyurl.com/z
  url='u3'   salt=0 code='R'  ô trống -> cấp mới
=> encode('u3') = http://tinyurl.com/R
  url='u4'   salt=0 code='I'  ô trống -> cấp mới
=> encode('u4') = http://tinyurl.com/I
  url='u5'   salt=0 code='Z'  ô trống -> cấp mới
=> encode('u5') = http://tinyurl.com/Z
  url='u6'   salt=0 code='a'  đụng độ với 'u1' -> salt+1
  url='u6'   salt=1 code='C'  ô trống -> cấp mới
=> encode('u6') = http://tinyurl.com/C
  url='u7'   salt=0 code='o'  ô trống -> cấp mới
=> encode('u7') = http://tinyurl.com/o
  url='u8'   salt=0 code='f'  ô trống -> cấp mới
=> encode('u8') = http://tinyurl.com/f
  url='u9'   salt=0 code='Y'  ô trống -> cấp mới
=> encode('u9') = http://tinyurl.com/Y
  url='u10'  salt=0 code='F'  ô trống -> cấp mới
=> encode('u10

## 9. Code nộp LeetCode

Ta nộp **Cách 2** (bộ đếm + dict): ngắn nhất, không có đụng độ, mọi thao tác `O(1)`.

In [19]:
class Codec:
    def __init__(self):
        self.m = {}
        self.idx = 0
        self.domain = "https://tinyurl.com/"

    def encode(self, longUrl: str) -> str:
        """Encodes a URL to a shortened URL."""
        self.idx += 1
        self.m[str(self.idx)] = longUrl
        return f"{self.domain}{self.idx}"

    def decode(self, shortUrl: str) -> str:
        """Decodes a shortened URL to its original URL."""
        return self.m[shortUrl.rsplit("/", 1)[-1]]


# Your Codec object will be instantiated and called as such:
# codec = Codec()
# codec.decode(codec.encode(url))
tiny, ans = leetcode_driver(Codec, URL0)
print(tiny, "->", ans)
assert ans == URL0

https://tinyurl.com/1 -> https://leetcode.com/problems/design-tinyurl


## 10. Vì sao đúng

Phát biểu **bất biến**: *sau mỗi lần gọi, với mọi mã `c` có trong `m`, `m[c]` là URL mà lần `encode` đã cấp `c` nhận được, và **chưa từng bị ghi đè**.*

- **Cách 2 (đếm):** mỗi `encode` dùng `idx` mới (đã tăng) làm khóa; vì `idx` **chỉ tăng**, khóa này chưa từng dùng ⇒ không ghi đè khóa cũ ⇒ bất biến giữ. `decode` đọc đúng `m[c]` ⇒ trả đúng URL.
- **Cách 3 (ngẫu nhiên):** vòng `while` chỉ thoát khi `code not in m`; sau đó mới gán ⇒ không ghi đè ⇒ bất biến giữ. Tính ngẫu nhiên chỉ ảnh hưởng **độ dài thời gian chờ** (số lần thử), không ảnh hưởng độ đúng.
- **Cách 4 (băm):** `encode` chỉ gán `m[code]` khi `code` chưa có; nếu có thì hoặc đúng URL này (trả mã cũ, trạng thái không đổi) hoặc URL khác (đổi `salt`). Mỗi URL luôn dò theo **cùng thứ tự salt**, nên lần gọi sau thấy lại đúng ô đã cấp. Vòng lặp **kết thúc** vì mỗi lần đụng độ ứng với một ô **đã bị chiếm** và có hữu hạn ô bị chiếm, trong khi mã theo `salt` khác nhau gần như độc lập — xác suất liên tiếp đụng độ giảm theo hàm mũ khi không gian mã chưa đầy.
- **Cách 5 (không trạng thái):** `zlib.decompress(zlib.compress(x)) == x` và base64 (có bù đệm) đảo ngược chính xác ⇒ `decode(encode(u)) == u` mà không cần nhớ.
- **Phần phân tích mã:** các cách có trạng thái dùng `rsplit("/", 1)[-1]`; mã chỉ gồm `0-9a-zA-Z` (không có `/`) nên cắt đúng; Cách 5 dùng bảng chữ **url-safe** (`-` `_`) vì cùng lý do.
- **Bao gồm URL tùy ý:** vì ta coi URL là **chuỗi mờ** (không phân tích bên trong), mọi `url` dài ≤ `10^4`, có `/`, `?`, `#`, thậm chí chứa cả một "tinyurl" khác đều xử lý đúng.

Các lập luận trên được kiểm chứng bằng bộ test mục 12 (round-trip, tính duy nhất, ép đụng độ, thống kê).

## 11. Phân tích Time / Space Complexity

Gọi `L` là độ dài URL (`≤ 10^4`), `n` là số URL đã mã hóa.

| Cách | `encode` | `decode` | Space | Ghi chú |
|---|---|---|---|---|
| 1. List tuyến tính | `O(n·L)` xấu nhất | `O(1)` | `O(n·L)` | dedupe nhưng chậm dần |
| 2. Bộ đếm + dict | `O(L)` | `O(1)` | `O(n·L)` | **nộp bài**, không đụng độ |
| 3. Ngẫu nhiên + thử lại | `O(L)` kỳ vọng | `O(1)` | `O(n·L)` | cần xử lý đụng độ, mã không đoán được |
| 4. Băm + salt | `O(L)` kỳ vọng | `O(1)` | `O(n·L)` | cùng URL → cùng short URL |
| 5. Không trạng thái | `O(L)` | `O(L)` | `O(1)` trạng thái | short URL dài, giải mã được bởi đối tượng khác |

Với `L ≤ 10^4` và số lệnh vừa phải, mọi cách đều nhanh; khác biệt nằm ở **tính chất thiết kế** (đụng độ, độ dài mã, đoán mã, dedupe, trạng thái), không phải tốc độ — trừ Cách 1 chậm dần theo `n`. Mục 12 có đo thời gian.

## 12. Bộ test

Vì bài có **nhiều đáp án đúng**, ta kiểm tra bằng **tính chất** (không so short URL cụ thể). Hàm `check_codec(cls, urls)` kiểm:

1. **Kiểu & hình dạng:** `encode` trả `str` không rỗng, bắt đầu bằng `http`, không có khoảng trắng.
2. **Round-trip trên cùng đối tượng:** `decode(encode(u)) == u` ngay sau khi `encode`.
3. **Không đè nhau:** `encode` **nhiều** URL trước, rồi `decode` **theo thứ tự xáo trộn** — mọi URL vẫn đúng.
4. **Tính duy nhất:** các URL **khác nhau** phải ra short URL **khác nhau** (nếu không, `decode` không thể đúng cho cả hai).
5. **URL lặp:** `encode` cùng một URL nhiều lần: **mọi** short URL đều `decode` về URL đó.

Rồi: ví dụ đề, edge case (URL 1 ký tự, URL dài `10^4`, URL chứa `/`, `?`, `#`, URL trông như short URL, Unicode), **ngẫu nhiên có seed**, **hai đối tượng độc lập**, **thống kê** (mã ngẫu nhiên đều ký tự), **ép đụng độ** (Cách 3, 4), **đo thời gian**.

In [20]:
IMPLS = {
    "list": CodecList,
    "counter": CodecCounter,
    "random": CodecRandom,
    "hash": CodecHash,
    "stateless": CodecStateless,
    "Codec": Codec,
}


def check_codec(cls, urls, seed=0):
    codec = cls()
    shorts = []
    for u in urls:
        s = codec.encode(u)
        assert isinstance(s, str) and s and s.startswith("http") and not any(ch.isspace() for ch in s), s
        assert codec.decode(s) == u, "round-trip ngay lập tức sai"
        shorts.append(s)
    # decode theo thứ tự xáo trộn: không có URL nào bị đè
    order = list(range(len(urls)))
    random.Random(seed).shuffle(order)
    for i in order:
        assert codec.decode(shorts[i]) == urls[i], ("bị đè/sai", urls[i][:50])
    # URL khác nhau -> short khác nhau
    seen = {}
    for u, s in zip(urls, shorts):
        assert seen.setdefault(s, u) == u, "hai URL khác nhau cùng một short URL"
    return shorts


# 1) Ví dụ đề trên mọi cách
for name, cls in IMPLS.items():
    tiny, ans = leetcode_driver(cls, URL0)
    assert ans == URL0, name
print("Ví dụ đề (6 cài đặt): OK")

Ví dụ đề (6 cài đặt): OK


In [21]:
# 2) Edge case
long_url = "https://example.com/" + "a" * (10**4 - len("https://example.com/"))
assert len(long_url) == 10**4
edge_urls = [
    "h",                                                  # URL 1 ký tự (độ dài tối thiểu)
    "http://a.b",
    "https://example.com/path/to/page?x=1&y=2#frag",
    "https://example.com/",
    "https://example.com",                                # chỉ khác dấu / cuối
    "HTTPS://EXAMPLE.COM/",                               # khác hoa/thường
    "http://tinyurl.com/1",                               # URL trông như short URL
    "http://tinyurl.com/",
    "https://example.com/a/b/c/d/e/f/g/h/i/j",
    "https://example.com/%E2%9C%93?q=a+b%20c",
    "https://example.com/ünï/文字",                       # Unicode (an toàn thêm)
    long_url,                                             # độ dài tối đa 10^4
    long_url[:-1] + "b",                                  # khác đúng ký tự cuối
    "https://leetcode.com/problems/design-tinyurl",
]
for name, cls in IMPLS.items():
    check_codec(cls, edge_urls)
    check_codec(cls, edge_urls + edge_urls)               # mã hóa lặp mọi URL lần hai
print(f"Edge case: OK ({len(edge_urls)} URL x 6 cài đặt, gồm URL độ dài 10^4)")

Edge case: OK (14 URL x 6 cài đặt, gồm URL độ dài 10^4)


In [22]:
# 3) Ngẫu nhiên có seed: URL hợp lệ độ dài 1..10^4, có lặp
def gen_url(rng):
    scheme = rng.choice(["http", "https"])
    host = ".".join("".join(rng.choices(string.ascii_lowercase, k=rng.randint(1, 8))) for _ in range(rng.randint(2, 3)))
    segs = ["".join(rng.choices(string.ascii_letters + string.digits + "-._~%", k=rng.randint(0, 12)))
            for _ in range(rng.randint(0, 6))]
    url = f"{scheme}://{host}/" + "/".join(segs)
    if rng.random() < 0.5:
        url += "?" + "&".join(f"{rng.choice('abc')}={rng.randint(0, 99)}" for _ in range(rng.randint(1, 3)))
    if rng.random() < 0.2:
        url += "#" + "".join(rng.choices(string.ascii_lowercase, k=5))
    if rng.random() < 0.05:                                # thỉnh thoảng URL rất dài
        url += "/" + "x" * rng.randint(1, 9000)
    return url[:10**4]


rng = random.Random(535)
pool = [gen_url(rng) for _ in range(800)]
batch = [rng.choice(pool) for _ in range(2000)]            # có lặp
assert all(1 <= len(u) <= 10**4 for u in batch)
for name, cls in IMPLS.items():
    check_codec(cls, batch, seed=1)
print(f"Ngẫu nhiên: {len(batch)} lệnh encode ({len(set(batch))} URL khác nhau) x 6 cài đặt: OK")

Ngẫu nhiên: 2000 lệnh encode (732 URL khác nhau) x 6 cài đặt: OK


In [23]:
# 3b) Với URL lặp: mọi short URL của cùng một URL đều decode đúng; cài đặt idempotent cho cùng short URL
dup = "https://example.com/dup"
for name, cls in IMPLS.items():
    c = cls()
    shorts = [c.encode(dup) for _ in range(5)]
    assert all(c.decode(s) == dup for s in shorts)
    kind = "cùng short URL" if len(set(shorts)) == 1 else f"{len(set(shorts))} short URL khác nhau"
    print(f"{name:<10} encode 5 lần cùng URL -> {kind}")
    if name in ("list", "hash", "stateless"):              # thiết kế có tính idempotent
        assert len(set(shorts)) == 1
    if name == "counter" or name == "Codec":
        assert len(set(shorts)) == 5                       # đếm: mỗi lần một mã mới (vẫn đúng đề)

list       encode 5 lần cùng URL -> cùng short URL
counter    encode 5 lần cùng URL -> 5 short URL khác nhau
random     encode 5 lần cùng URL -> 5 short URL khác nhau
hash       encode 5 lần cùng URL -> cùng short URL
stateless  encode 5 lần cùng URL -> cùng short URL
Codec      encode 5 lần cùng URL -> 5 short URL khác nhau


In [24]:
# 4) Hai đối tượng độc lập; đối tượng mới KHÔNG biết dữ liệu của đối tượng cũ (trừ Cách 5)
for name, cls in IMPLS.items():
    a, b = cls(), cls()
    ta = a.encode("https://a.example/x")
    tb = b.encode("https://b.example/y")
    assert a.decode(ta) == "https://a.example/x" and b.decode(tb) == "https://b.example/y"
    try:
        out = b.decode(ta)
        note = f"decode chéo trả {out!r}"                  # có thể trả URL khác hoặc đúng (stateless)
    except (KeyError, IndexError):
        note = "decode chéo -> lỗi (đúng: đề chỉ đảm bảo cùng một đối tượng)"
    print(f"{name:<10} {note}")
assert CodecStateless().decode(CodecStateless().encode(URL0)) == URL0     # stateless: đối tượng mới vẫn giải mã

list       decode chéo trả 'https://b.example/y'
counter    decode chéo trả 'https://b.example/y'
random     decode chéo -> lỗi (đúng: đề chỉ đảm bảo cùng một đối tượng)
hash       decode chéo -> lỗi (đúng: đề chỉ đảm bảo cùng một đối tượng)
stateless  decode chéo trả 'https://a.example/x'
Codec      decode chéo trả 'https://b.example/y'


In [25]:
# 5) Thống kê Cách 3: 20000 mã ngẫu nhiên, mỗi ký tự trong bảng 62 ký tự xuất hiện đều (<= 5 sigma)
c = CodecRandom(seed=2026)
cnt = Counter()
codes = []
for i in range(20000):
    s = c.encode(f"https://x.example/{i}")
    code_i = s.rsplit("/", 1)[-1]
    assert len(code_i) == 6 and set(code_i) <= set(ALPHABET)
    codes.append(code_i)
    cnt.update(code_i)
total = 20000 * 6
p = 1 / 62
sigma = (total * p * (1 - p)) ** 0.5
worst = max(abs(cnt[ch] - total * p) / sigma for ch in ALPHABET)
assert set(cnt) == set(ALPHABET) and worst < 5, worst
assert len(set(codes)) == 20000                            # mọi mã đều khác nhau (mọi URL khác nhau)
print(f"Thống kê: 62 ký tự đều xuất hiện, lệch tối đa {worst:.2f} sigma; 20000 mã khác nhau; số lần bốc lại = {c.retries}")

Thống kê: 62 ký tự đều xuất hiện, lệch tối đa 2.56 sigma; 20000 mã khác nhau; số lần bốc lại = 0


In [26]:
# 6) ÉP ĐỤNG ĐỘ: bảng nhỏ để nhánh "thử lại" / "đổi salt" chắc chắn chạy
class CodecRandom62(CodecRandom):
    LEN = 1                                                # chỉ 62 mã


for seed in range(5):
    c = CodecRandom62(seed)
    urls = [f"https://t.example/{i}" for i in range(60)]   # 60 URL trong 62 ô: đụng độ gần như chắc chắn
    shorts = [c.encode(u) for u in urls]
    assert all(c.decode(s) == u for s, u in zip(shorts, urls))
    assert len(set(shorts)) == 60
assert c.retries > 0
print("Cách 3 ép đụng độ: 5 seed x 60 URL trong 62 mã: mọi URL decode đúng; số lần bốc lại (seed 4):", c.retries)

c = CodecHashTiny()
urls = [f"https://t.example/{i}" for i in range(50)]
shorts = [c.encode(u) for u in urls]
assert all(c.decode(s) == u for s, u in zip(shorts, urls)) and len(set(shorts)) == 50
assert c.probes > 0
again = [c.encode(u) for u in urls]
assert again == shorts, "encode lại cùng URL phải ra đúng mã cũ dù đã từng đụng độ"
print("Cách 4 ép đụng độ: 50 URL trong 62 mã: đúng, idempotent; số lần đổi salt:", c.probes)

# bảng ĐẦY: random bốc lại vô hạn nên ta KHÔNG thử; hash cũng vậy -> trong thực tế tăng LEN trước khi đầy
print("sức chứa thật: 62^6 =", 62**6)

Cách 3 ép đụng độ: 5 seed x 60 URL trong 62 mã: mọi URL decode đúng; số lần bốc lại (seed 4): 99
Cách 4 ép đụng độ: 50 URL trong 62 mã: đúng, idempotent; số lần đổi salt: 102
sức chứa thật: 62^6 = 56800235584


In [27]:
# 7) Thời gian: Cách 1 chậm dần (bậc hai), các cách khác gần tuyến tính
def time_cls(cls, n):
    c = cls()
    urls = [f"https://perf.example/{i}/" + "p" * 50 for i in range(n)]
    t0 = time.perf_counter()
    shorts = [c.encode(u) for u in urls]
    for u, s in zip(urls, shorts):
        assert c.decode(s) == u
    return (time.perf_counter() - t0) * 1000


for n in (2000, 4000, 8000):
    row = {name: time_cls(cls, n) for name, cls in IMPLS.items() if name in ("list", "counter", "random", "hash", "stateless")}
    print(f"n={n:>5}: " + " | ".join(f"{k} {v:7.1f} ms" for k, v in row.items()))

n= 2000: list    16.2 ms | counter     0.8 ms | random     2.0 ms | hash     4.1 ms | stateless    26.3 ms


n= 4000: list   107.1 ms | counter     2.1 ms | random     4.8 ms | hash    11.8 ms | stateless    30.5 ms


n= 8000: list   294.6 ms | counter     3.5 ms | random     8.5 ms | hash    19.9 ms | stateless    48.8 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng base64 **chuẩn** (có `/` và `+`) rồi cắt bằng `rsplit("/")`

Bảng chữ cái base64 chuẩn gồm `/` — nếu mã chứa `/` thì `rsplit("/", 1)[-1]` chỉ lấy được **đoạn cuối**, mất phần đầu của mã.

In [28]:
class WrongStdB64:
    DOMAIN = "http://tinyurl.com/"

    def encode(self, longUrl):
        return self.DOMAIN + base64.b64encode(longUrl.encode()).decode()      # b64encode chuẩn: có "/" và "+"

    def decode(self, shortUrl):
        return base64.b64decode(shortUrl.rsplit("/", 1)[-1]).decode()


u = "https://example.com/?q=ß???"                         # nhiều khả năng sinh ký tự "/" trong base64
bad = WrongStdB64()
s = bad.encode(u)
print(s)
try:
    print("kết quả:", bad.decode(s) == u)
except Exception as e:
    print("lỗi:", type(e).__name__, e)

# kiểm tra có tồn tại URL làm lỗi ngay không
rng = random.Random(1)
fails = 0
for _ in range(300):
    uu = gen_url(rng) + "?" + "".join(rng.choices("?>~", k=6))
    try:
        fails += bad.decode(bad.encode(uu)) != uu
    except Exception:
        fails += 1
print("số URL ngẫu nhiên bị sai/lỗi trong 300:", fails)
assert fails > 0
assert CodecStateless().decode(CodecStateless().encode(u)) == u      # bản url-safe đúng

http://tinyurl.com/aHR0cHM6Ly9leGFtcGxlLmNvbS8/cT3Dnz8/Pw==
kết quả: False
số URL ngẫu nhiên bị sai/lỗi trong 300: 219


Cách sửa: dùng `urlsafe_b64encode` (bảng `-_`), hoặc chọn bảng chữ chỉ gồm `0-9a-zA-Z`, và luôn nghĩ "**mã có thể chứa ký tự dùng để cắt không?**".

### Lỗi 2: băm rồi cắt ngắn **mà không xử lý đụng độ**

Lấy `sha256(url)` rồi giữ vài ký tự đầu làm mã, ghi `m[code] = url`. Hai URL khác nhau cho cùng mã sẽ **đè nhau**, `decode` trả sai URL.

In [29]:
class WrongHashNoCollision:
    DOMAIN = "http://tinyurl.com/"
    LEN = 1                                                # mã ngắn để đụng độ lộ ra sớm (với mã 6 cũng sai, chỉ hiếm hơn)

    def __init__(self):
        self.m = {}

    def encode(self, longUrl):
        h = hashlib.sha256(longUrl.encode()).hexdigest()[: self.LEN]
        self.m[h] = longUrl                                # ghi đè khi trùng!
        return self.DOMAIN + h

    def decode(self, shortUrl):
        return self.m[shortUrl.rsplit("/", 1)[-1]]


bad = WrongHashNoCollision()
urls = [f"https://t.example/{i}" for i in range(40)]       # 40 URL trong 16 mã hex
shorts = [bad.encode(u) for u in urls]
wrong = sum(bad.decode(s) != u for s, u in zip(shorts, urls))
print("sai:", wrong, "/", len(urls), "URL decode ra URL khác")
assert wrong > 0

good = CodecHashTiny()
shorts = [good.encode(u) for u in urls]
assert all(good.decode(s) == u for s, u in zip(shorts, urls))
print("đúng: CodecHash xử lý đụng độ ->", sum(good.decode(s) != u for s, u in zip(shorts, urls)), "URL sai")

sai: 29 / 40 URL decode ra URL khác
đúng: CodecHash xử lý đụng độ -> 0 URL sai


### Lỗi 3: `decode` giả định mã có **độ dài cố định** (ví dụ `shortUrl[-1]`, `shortUrl[-6:]`)

Bộ đếm cho id `1, 2, ..., 9, 10, 11, ...`: từ 10 trở đi mã có 2 chữ số. Cắt cố định 1 ký tự cuối sẽ chỉ đọc "0" của "10".

In [30]:
class WrongFixedWidth(CodecCounter):
    def decode(self, shortUrl):
        return self.m[shortUrl[-1]]                         # sai: chỉ lấy 1 ký tự cuối


bad = WrongFixedWidth()
urls = [f"https://t.example/{i}" for i in range(12)]
shorts = [bad.encode(u) for u in urls]
bad_out = []
for u, s in zip(urls, shorts):
    try:
        bad_out.append(bad.decode(s) == u)
    except KeyError:
        bad_out.append(False)
print("kết quả từng URL (False = sai):", bad_out)
assert bad_out[:9] == [True] * 9 and not all(bad_out)

good = CodecCounter()
assert all(good.decode(s) == u for s, u in zip([good.encode(u) for u in urls], urls))

kết quả từng URL (False = sai): [True, True, True, True, True, True, True, True, True, False, False, False]


Cách sửa: **tách theo dấu phân cách** (`rsplit("/", 1)[-1]`) hoặc làm mã **đệm cố định độ dài** ngay từ `encode` (`str(idx).zfill(6)`), và dùng đúng hằng số đó ở `decode`.

### Lỗi 4: dùng `hash()` có sẵn của Python làm mã

`hash("abc")` thay đổi **giữa các lần chạy chương trình** (do hash randomization) và **không đảo ngược** được. Bài tập này chạy trong **cùng một đối tượng** nên nhiều lúc vẫn "qua" nhưng mã không ổn định, và vẫn cần bảng `code → url` để `decode`.

In [31]:
def hash_in_fresh_process(seed):
    out = subprocess.run(
        [sys.executable, "-c", "print(hash('https://leetcode.com'))"],
        env={"PYTHONHASHSEED": str(seed)}, capture_output=True, text=True, check=True)
    return out.stdout.strip()


h1, h2 = hash_in_fresh_process(1), hash_in_fresh_process(2)
print("hash(...) ở hai tiến trình:", h1, h2)
assert h1 != h2
print("hashlib.sha256 luôn ổn định:", hashlib.sha256(b"https://leetcode.com").hexdigest()[:12])

hash(...) ở hai tiến trình: 6450495843195221838 -6870019210805215182
hashlib.sha256 luôn ổn định: 55a389265834


### Lỗi 5: để bảng `dict` ở **cấp lớp** thay vì `self` trong `__init__`

`store = {}` viết trực tiếp trong thân lớp là **một dict dùng chung cho mọi đối tượng**. Khi LeetCode tạo `Codec()` mới cho mỗi bộ test, dữ liệu và bộ đếm cũ **rò sang** bộ test sau.

In [32]:
class WrongClassLevelState:
    store = {}                                              # dùng chung giữa MỌI đối tượng!
    idx = 0

    def encode(self, longUrl):
        WrongClassLevelState.idx += 1
        self.store[str(self.idx)] = longUrl
        return f"http://tinyurl.com/{self.idx}"

    def decode(self, shortUrl):
        return self.store[shortUrl.rsplit("/", 1)[-1]]


first = WrongClassLevelState()
first.encode("https://old.example")
second = WrongClassLevelState()                             # "đối tượng mới" nhưng vẫn thấy dữ liệu cũ
print("bộ nhớ của đối tượng mới:", second.store)
assert second.store != {}                                   # lỗi: đáng lẽ phải rỗng

ok = CodecCounter()
assert ok.m == {} and ok.idx == 0
WrongClassLevelState.store.clear()
WrongClassLevelState.idx = 0

bộ nhớ của đối tượng mới: {'1': 'https://old.example'}


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** đề "thiết kế `encode`/`decode` đảo ngược của nhau, không ràng buộc cách làm" ⇒ **bảng ánh xạ + khóa duy nhất**. Hỏi ngay: "khóa lấy từ đâu?" — đếm (an toàn), ngẫu nhiên/băm (cần xử lý đụng độ), hay chính dữ liệu (không trạng thái).
- **Bài thiết kế nhiều đáp án:** tự viết **bộ kiểm tra tính chất** (round-trip, tính duy nhất, độc lập đối tượng) thay vì so đầu ra. Cùng cách tư duy cho *Insert Delete GetRandom*, *LRU Cache*, *Serialize and Deserialize Binary Tree*, *Encode and Decode Strings*.
- **Đụng độ luôn xảy ra** khi mã ngắn so với số lượng: dùng **nghịch lý ngày sinh** ước lượng (`n² / 2N`), và luôn có nhánh "đã có rồi thì thử lại".
- **Đếm là khóa miễn phí:** nếu không cần mã khó đoán, bộ đếm + dict là lời giải ngắn nhất và nhanh nhất; chuyển sang base-62 để mã ngắn hơn (mục 15).
- **Cắt chuỗi theo dấu phân cách**, không theo vị trí cố định; và đảm bảo mã **không chứa** dấu phân cách.
- **Test nhánh hiếm bằng cách thu nhỏ không gian:** muốn thấy đụng độ, giảm `LEN` về 1 hoặc bảng chữ cái về 2 ký tự (mục 12.6) thay vì chờ may rủi.
- Trong phỏng vấn thiết kế hệ thống: nhắc tới **ID tăng dần phân tán** (Snowflake), **base-62**, **cơ sở dữ liệu** thay `dict`, **cache**, **hết hạn**, **thống kê lượt nhấp**.

## 15. Biến thể của bài

1. **Mã base-62 theo bộ đếm:** thay `str(idx)` bằng `to_base62(idx)` ⇒ mã ngắn hơn rất nhiều (10⁹ URL chỉ cần 5–6 ký tự); `decode` dùng `from_base62` hoặc tra dict trực tiếp.
2. **Custom alias:** người dùng chọn mã (`tinyurl.com/my-blog`); trùng mã thì từ chối (bài tập 5).
3. **Dedupe hai chiều:** thêm dict `url → code` để cùng URL luôn ra cùng short URL (như Cách 1/4 nhưng `O(1)`).
4. **Đếm lượt truy cập, hết hạn (TTL), xóa short URL.**
5. **Phân tán nhiều máy:** bộ đếm tập trung là điểm nghẽn ⇒ cấp phát theo đoạn (range) cho từng máy, hoặc Snowflake ID.
6. **Bài liên quan:** LeetCode 297 (Serialize and Deserialize Binary Tree), 271 (Encode and Decode Strings), 380 (Insert Delete GetRandom O(1)).

Cài nhanh biến thể 1 + 3 (base-62, dedupe hai chiều, đếm lượt truy cập) và kiểm tra bằng bộ test của mục 12:

In [33]:
class CodecBase62:
    DOMAIN = "http://tinyurl.com/"

    def __init__(self):
        self.code2url = {}
        self.url2code = {}
        self.visits = Counter()
        self.idx = 0

    def encode(self, longUrl):
        code = self.url2code.get(longUrl)
        if code is None:
            self.idx += 1
            code = to_base62(self.idx)
            self.code2url[code] = longUrl
            self.url2code[longUrl] = code
        return self.DOMAIN + code

    def decode(self, shortUrl):
        code = shortUrl.rsplit("/", 1)[-1]
        self.visits[code] += 1
        return self.code2url[code]


check_codec(CodecBase62, edge_urls + batch, seed=3)
c = CodecBase62()
s = [c.encode(f"https://v.example/{i}") for i in range(100000)]
assert s[0].endswith("/1") and s[34].endswith("/z") and s[60].endswith("/Z") and s[61].endswith("/10")
assert len(s[-1].rsplit("/", 1)[-1]) == 3
assert c.encode("https://v.example/5") == s[5]             # dedupe hai chiều
c.decode(s[7]); c.decode(s[7]); c.decode(s[8])
print("100000 URL -> mã dài tối đa", max(len(x.rsplit("/", 1)[-1]) for x in s), "ký tự; lượt truy cập:", dict(c.visits))

100000 URL -> mã dài tối đa 3 ký tự; lượt truy cập: {'8': 2, '9': 1}


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time `encode` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. List tuyến tính | số dòng trong sổ, dedupe bằng quét | `O(n·L)` | `O(n·L)` | chỉ để minh họa/baseline |
| 2. Bộ đếm + dict | khóa duy nhất bằng cấu tạo | `O(L)` | `O(n·L)` | **làm bài**, thực tế khi không cần mã khó đoán |
| 3. Ngẫu nhiên + thử lại | khóa duy nhất bằng kiểm tra | `O(L)` kỳ vọng | `O(n·L)` | cần mã **khó đoán**; chú ý khi bảng gần đầy |
| 4. Băm + salt | mã theo nội dung, dò khi đụng độ | `O(L)` kỳ vọng | `O(n·L)` | cần **cùng URL → cùng mã**, không cần dict `url → code` |
| 5. Không trạng thái | mã chứa URL (nén + base64) | `O(L)` | `O(1)` trạng thái | nhiều máy không chia sẻ trạng thái; không cần mã ngắn |

Trực giác chọn: **đếm** là mặc định; chuyển sang **ngẫu nhiên** khi cần che giấu thứ tự; sang **băm** khi muốn tính chất idempotent từ nội dung; **không trạng thái** chỉ khi không thể lưu gì.

## 17. 5 bài tập vận dụng (dễ → khó)

Mọi cell TODO **không gọi** hàm khi chạy notebook; đáp án có assert ở mục 19.

### Bài 1 (đoán output)

Đoán trước, rồi chạy để kiểm tra.

```python
c = CodecCounter()
a = c.encode("x.com")
b = c.encode("y.com")
d = c.encode("x.com")
print(a, b, d)
print(c.decode(d), c.decode(a) == c.decode(d))
```

In [34]:
c_b1 = CodecCounter()
a = c_b1.encode("x.com")
b = c_b1.encode("y.com")
d = c_b1.encode("x.com")
print(a, b, d)
print(c_b1.decode(d), c_b1.decode(a) == c_b1.decode(d))

http://tinyurl.com/1 http://tinyurl.com/2 http://tinyurl.com/3
x.com True


### Bài 2 (tính tay)

Với bảng `ALPHABET = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"`, hãy tính **bằng tay** `to_base62(1000)` (gợi ý: chia 1000 cho 62 lấy thương và dư, đổi mỗi số dư thành ký tự theo chỉ số trong `ALPHABET`). Ghi vào `my_answer_2` (để `None` đến khi bạn tính xong).

In [35]:
my_answer_2 = None          # TODO: điền chuỗi bạn tính tay, ví dụ "ab"

### Bài 3 (điền TODO)

Hoàn thành lớp `CodecTodo`: `encode` đã viết sẵn; bạn viết `decode` — lấy mã sau dấu `/` cuối và tra trong `self.m`. Cell bên dưới **chỉ định nghĩa**, không gọi.

In [36]:
class CodecTodo:
    def __init__(self):
        self.m = {}
        self.idx = 0

    def encode(self, longUrl):
        self.idx += 1
        self.m[str(self.idx)] = longUrl
        return f"http://tinyurl.com/{self.idx}"

    def decode(self, shortUrl):
        # TODO: lấy mã sau dấu "/" cuối rồi tra self.m
        pass

### Bài 4 (tự viết lại)

Tự viết lớp `MyBidirectionalCodec` theo hướng **dedupe hai chiều** (hai dict `url → code` và `code → url`, mã là số thứ tự) sao cho **encode cùng URL hai lần cho cùng short URL**. Không nhìn mục 15. Cell dưới chỉ để khung rỗng.

In [37]:
class MyBidirectionalCodec:
    # TODO: viết __init__, encode, decode
    pass

### Bài 5 (biến thể / mini challenge)

**Custom alias:** viết lớp `AliasCodec` với `encode(longUrl, alias=None)`:

- Nếu `alias` là `None`: cấp mã tự động (ví dụ bộ đếm `"auto1"`, `"auto2"`, ... — và **đảm bảo chưa dùng**).
- Nếu `alias` cho trước và **chưa bị dùng**: dùng `alias` làm mã.
- Nếu `alias` **đã bị URL khác dùng**: ném `ValueError`. Nếu bị chính URL này dùng: trả lại mã cũ.
- `decode` tra mã sau dấu `/` cuối.

In [38]:
class AliasCodec:
    # TODO: viết __init__, encode(longUrl, alias=None), decode(shortUrl)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: `encode` luôn tăng `idx`, kể cả URL lặp.
- Bài 2: `divmod(1000, 62)` cho thương và dư; lặp lại với thương.
- Bài 3: dùng `rsplit("/", 1)[-1]` để lấy mã.
- Bài 4: cần hai dict và một bộ đếm; khi URL đã có thì **không** tăng bộ đếm.
- Bài 5: nhớ hai dict: `code → url` và `url → code`, để kiểm tra "alias đã bị dùng bởi URL nào".

**Mức 2 (vừa):**
- Bài 1: ba short URL là `/1`, `/2`, `/3` (domain `http://tinyurl.com/`); `decode(d)` là `"x.com"`.
- Bài 2: `1000 = 16·62 + 8`; số dư 8 → `ALPHABET[8]`, thương 16 → `ALPHABET[16]`; viết thương trước, dư sau.
- Bài 3: `return self.m[shortUrl.rsplit("/", 1)[-1]]`.
- Bài 4: `encode`: `if longUrl in self.url2code: return domain + self.url2code[longUrl]`; còn lại cấp id mới.
- Bài 5: với `alias=None`, vòng `while f"auto{k}" in self.code2url: k += 1`.

**Mức 3 (gần đáp án):**
- Bài 1: in ra `http://tinyurl.com/1 http://tinyurl.com/2 http://tinyurl.com/3`, rồi `x.com True`.
- Bài 2: ký tự thứ 16 là `"g"` (`0-9` là 0..9, `a`=10, ..., `g`=16), ký tự thứ 8 là `"8"` ⇒ `"g8"`.
- Bài 3: một dòng `return` như mức 2.
- Bài 4: ghi cả hai dict khi cấp mới: `self.code2url[code] = url; self.url2code[url] = code`.
- Bài 5: `if alias in self.code2url: if self.code2url[alias] == longUrl: return ...; raise ValueError`.

## 19. Đáp án tham khảo

Phần này đặt cuối bài giảng; mọi đáp án có assert.

In [39]:
# Bài 1
assert (a, b, d) == ("http://tinyurl.com/1", "http://tinyurl.com/2", "http://tinyurl.com/3")
assert c_b1.decode(d) == "x.com" and c_b1.decode(a) == c_b1.decode(d)
print("Bài 1:", a, b, d)

# Bài 2
ans_2 = "g8"
assert to_base62(1000) == ans_2 == ALPHABET[16] + ALPHABET[8]
assert 16 * 62 + 8 == 1000
print("Bài 2:", ans_2, "(1000 = 16*62 + 8)")


# Bài 3
class CodecTodoSol(CodecTodo):
    def decode(self, shortUrl):
        return self.m[shortUrl.rsplit("/", 1)[-1]]


c = CodecTodoSol()
shorts = [c.encode(f"https://t.example/{i}") for i in range(30)]
assert all(c.decode(s) == f"https://t.example/{i}" for i, s in enumerate(shorts))
print("Bài 3: OK")

Bài 1: http://tinyurl.com/1 http://tinyurl.com/2 http://tinyurl.com/3
Bài 2: g8 (1000 = 16*62 + 8)
Bài 3: OK


In [40]:
# Bài 4
class MyBidirectionalCodecSol:
    DOMAIN = "http://tinyurl.com/"

    def __init__(self):
        self.code2url = {}
        self.url2code = {}
        self.idx = 0

    def encode(self, longUrl):
        if longUrl in self.url2code:
            return self.DOMAIN + self.url2code[longUrl]
        self.idx += 1
        code = str(self.idx)
        self.code2url[code] = longUrl
        self.url2code[longUrl] = code
        return self.DOMAIN + code

    def decode(self, shortUrl):
        return self.code2url[shortUrl.rsplit("/", 1)[-1]]


check_codec(MyBidirectionalCodecSol, edge_urls + batch, seed=4)
c = MyBidirectionalCodecSol()
assert c.encode("a.com") == c.encode("a.com") != c.encode("b.com")
assert c.idx == 2                                           # URL lặp không làm tăng bộ đếm
print("Bài 4: OK")


# Bài 5
class AliasCodecSol:
    DOMAIN = "http://tinyurl.com/"

    def __init__(self):
        self.code2url = {}
        self.url2code = {}
        self.k = 0

    def encode(self, longUrl, alias=None):
        if alias is None:
            if longUrl in self.url2code:
                return self.DOMAIN + self.url2code[longUrl]
            self.k += 1
            while f"auto{self.k}" in self.code2url:
                self.k += 1
            alias = f"auto{self.k}"
        elif alias in self.code2url:
            if self.code2url[alias] == longUrl:
                return self.DOMAIN + alias
            raise ValueError(f"alias {alias!r} đã được dùng")
        self.code2url[alias] = longUrl
        self.url2code.setdefault(longUrl, alias)
        return self.DOMAIN + alias

    def decode(self, shortUrl):
        return self.code2url[shortUrl.rsplit("/", 1)[-1]]


c = AliasCodecSol()
s1 = c.encode("https://blog.example", alias="my-blog")
assert s1 == "http://tinyurl.com/my-blog" and c.decode(s1) == "https://blog.example"
assert c.encode("https://blog.example", alias="my-blog") == s1
try:
    c.encode("https://other.example", alias="my-blog")
    raise AssertionError("phải ném ValueError")
except ValueError as e:
    print("Bài 5: ValueError ->", e)
s2 = c.encode("https://auto.example")
assert c.decode(s2) == "https://auto.example"
try:
    c.encode("https://x.example", alias="auto1")           # "auto1" đã là mã tự động của URL khác
    raise AssertionError("phải ném ValueError")
except ValueError:
    pass
check_codec(AliasCodecSol, edge_urls + batch, seed=5)
print("Bài 5: OK")

Bài 4: OK
Bài 5: ValueError -> alias 'my-blog' đã được dùng
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (Cách 2: bộ đếm + dict):**

```python
class Codec:
    def __init__(self):
        self.m = {}
        self.idx = 0

    def encode(self, longUrl: str) -> str:
        self.idx += 1
        self.m[str(self.idx)] = longUrl
        return "https://tinyurl.com/" + str(self.idx)

    def decode(self, shortUrl: str) -> str:
        return self.m[shortUrl.rsplit("/", 1)[-1]]
```

**Công thức nhớ:** *khóa duy nhất → `m[khóa] = url`; trả `domain + khóa`; `decode` cắt khóa sau dấu `/` cuối rồi tra `m`.*

**Checklist trước khi submit:**

- [ ] Trạng thái nằm trong `self` (trong `__init__`), **không** ở cấp lớp.
- [ ] Mỗi `encode` cấp **khóa chưa từng dùng** (đếm, hoặc có vòng kiểm tra `not in m` nếu ngẫu nhiên/băm).
- [ ] `decode` cắt mã bằng dấu phân cách (`rsplit("/", 1)[-1]`), không giả định độ dài cố định; mã không chứa `/`.
- [ ] Đúng tên lớp `Codec` và hai method `encode`, `decode` đúng chữ ký; không đổi tên tham số bắt buộc.
- [ ] Đã test round-trip cho nhiều URL **trên cùng một đối tượng**, kể cả URL lặp, URL dài `10^4`, URL 1 ký tự.
- [ ] Đã test nhánh hiếm (đụng độ) bằng cách thu nhỏ không gian mã, nếu dùng ngẫu nhiên/băm.
- [ ] Không dùng `hash()` có sẵn làm mã; nếu băm thì dùng `hashlib`.